In [ ]:
import xarray as xr
import numpy as np

In [ ]:
%matplotlib qt5

In [ ]:
inputpath_raw = '/data/cburgard/PREPARE_FORCING/PREPARE_PISCES/raw/'
inputpath_interim = '/data/cburgard/PREPARE_FORCING/PREPARE_PISCES/interim/'

In [ ]:
file_orig = xr.open_dataset(inputpath_raw + 'etopo2bedmap.nc')

In [ ]:
file_orig

In [ ]:
lat = ((1./30.)*(file_orig.jpjo)+1./30.-90    )*-1

In [ ]:
lon = (-180.)+1./30.*(file_orig.jpio)

In [ ]:
file_new = file_orig.copy()
file_new = file_new.rename({'jpjo': 'lat', 'jpio': 'lon'}).assign_coords({'lat':lat.values, 'lon':lon.values})

In [ ]:
file_new['depth'].plot()

In [ ]:
file_new.lon

In [ ]:
file_gebco = xr.open_dataset(inputpath_raw + 'GEBCO_2024_sub_ice_topo.nc')

In [ ]:
diff_lon = 180-179.997917

In [ ]:
diff_lon

In [ ]:
file_new.lon

In [ ]:
240/30

In [ ]:
file_gebco_resampled = file_gebco.sel(lon=file_gebco.lon[::8],lat=file_gebco.lat[::8])
file_gebco_resampled = file_gebco_resampled.reindex(lat=list(reversed(file_gebco_resampled.lat)))

In [ ]:
file_new.lat.values - file_gebco_resampled.lat.values

In [ ]:
file_new2 = file_orig.copy()

In [ ]:
file_new2['depth'] = xr.DataArray(data=file_gebco_resampled['elevation'].values, dims=file_orig.dims)

In [ ]:
file_new2['depth'] = file_new2['depth'].where(file_new2['depth'] < 0,0)

In [ ]:
file_new2.to_netcdf(inputpath_interim + 'gebco_2024_inetopo2bedmap_format.nc')

In [ ]:
file_new2_cut_Ant = file_new2['depth'].where((file_new2['depth'] > -2000) & (file_new2.jpjo > 4500))

In [ ]:
file_new_merged = file_new2.copy()
file_new_merged['depth'] = file_new2['depth'].where((file_new2['depth'] > -2000) & (file_new2.jpjo > 4500), file_orig['depth'])

In [ ]:
file_new_merged.to_netcdf(inputpath_interim + 'etopo2bedmap_mergedwith_gebco2024_for_Ant.nc')

In [ ]:
file_new_merged['depth'].plot()

In [ ]:
file_gebco_new = file_gebco.interp2({'lon':file_new.lon,'lat':file_new.lat})